# ComplianceGPT — Single Query Verification (Gold)

This notebook runs **one gold-set query** and runs the verifier.

Use this notebook for evaluation and debugging verifier behavior.

Not intended for committee demos.

In [ ]:
# --- Colab + Drive bootstrap ---
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys
PROJECT_ROOT = "/content/drive/MyDrive/ComplianceGPT_v2"
assert os.path.isdir(PROJECT_ROOT), f"PROJECT_ROOT not found: {PROJECT_ROOT}"

SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import compliancegpt
print("compliancegpt.__file__ =", compliancegpt.__file__)


In [ ]:
# --- Canonical paths (Drive) ---
import os

DATA_DIR = os.path.join(PROJECT_ROOT, "data")

REV5_CATALOG_PATH = os.path.join(DATA_DIR, "ccs", "nist800-53", "NIST_SP-800-53_rev5_catalog.jsonl")
REV4_CATALOG_PATH = os.path.join(DATA_DIR, "ccs", "nist800-53", "NIST_SP-800-53_rev4_catalog.jsonl")

REV5_GOLD_PATH = os.path.join(DATA_DIR, "gold_standard_datasets", "nist800-53", "nist_sp800-53_rev5_gold-set_100q.csv")
REV4_GOLD_PATH = os.path.join(DATA_DIR, "gold_standard_datasets", "nist800-53", "nist_sp800-53_rev4_gold-set_36q.csv")

QUR_REV5_PATH = os.path.join(DATA_DIR, "qur_outputs", "qur_rewrites_rev5.csv")
QUR_REV4_PATH = os.path.join(DATA_DIR, "qur_outputs", "qur_rewrites_rev4.csv")


In [ ]:
# --- Helpers ---
from typing import Any, Dict, List
import pandas as pd

def load_gold(framework_version: str) -> pd.DataFrame:
    path = REV5_GOLD_PATH if framework_version.lower() == "rev5" else REV4_GOLD_PATH
    df = pd.read_csv(path, encoding="latin1")
    if "id" in df.columns and "query_id" not in df.columns:
        df["query_id"] = df["id"].astype(str)
    if "question" in df.columns and "original_query" not in df.columns:
        df["original_query"] = df["question"].astype(str)
    return df

def get_gold_row(df: pd.DataFrame, qid: Any) -> Dict[str, Any]:
    qid_str = str(qid)
    m = df[df["query_id"].astype(str) == qid_str]
    if m.empty:
        raise KeyError(f"query_id not found in gold set: {qid_str}")
    return dict(m.iloc[0].to_dict())

def load_qur_map(framework_version: str) -> Dict[str, List[str]]:
    path = QUR_REV5_PATH if framework_version.lower() == "rev5" else QUR_REV4_PATH
    if not os.path.exists(path):
        return {}
    df = pd.read_csv(path)
    out: Dict[str, List[str]] = {}
    for qid, g in df.groupby(df["query_id"].astype(str)):
        g2 = g.sort_values("rewrite_rank")
        out[qid] = [str(x) for x in g2["rewritten_query"].tolist() if isinstance(x, str) and x.strip()]
    return out


In [ ]:
# --- Build pipeline ---
from compliancegpt.pipeline.pipeline import ComplianceGPTPipeline

def build_pipeline(framework_version: str) -> ComplianceGPTPipeline:
    fw = framework_version.lower().strip()
    ccs_path = REV5_CATALOG_PATH if fw == "rev5" else REV4_CATALOG_PATH
    return ComplianceGPTPipeline(
        framework_version=fw,
        ccs_path=ccs_path,
        use_qur=False,
        org_profile_path=None,
        verify_strict_extras=False,
        verify_strict_verbatim=True,
        verify_strict_version=False,
    )


In [ ]:
# === CONFIG ===
FRAMEWORK_VERSION = "rev5"
GOLD_QUERY_ID = 1
USE_QUR_PRECOMPUTED = True
TOP_K = 12
SHOW_FULL_CONTRACT = True


In [ ]:
pipe = build_pipeline(FRAMEWORK_VERSION)
gold_df = load_gold(FRAMEWORK_VERSION)
gold_row = get_gold_row(gold_df, GOLD_QUERY_ID)

qur_map = load_qur_map(FRAMEWORK_VERSION) if USE_QUR_PRECOMPUTED else {}
qid = str(gold_row["query_id"])
rewrites = qur_map.get(qid, [])

query = str(gold_row["original_query"]).strip()

print("query_id:", qid)
print("control_id:", gold_row.get("control_id"))
print("resolution_policy:", gold_row.get("resolution_policy"))
print("query:", query)
print("n_rewrites:", len(rewrites))


In [ ]:
out = pipe.answer(
    query,
    top_k=int(TOP_K),
    rewrites=rewrites if rewrites else None,
    gold_row=gold_row,
    use_generator=True,
    run_verify=True,   # ALWAYS verify in this notebook
)

c = out.get("contract", out)
print("status:", c.get("status"))
print("verifier_ran:", c.get("verifier_ran", True))
print("verifier_pass:", c.get("verifier_pass"))
print("verifier_errors:", c.get("verifier_errors") or [])

if SHOW_FULL_CONTRACT:
    import json
    print(json.dumps(c, ensure_ascii=False, indent=2)[:8000])
